In [1]:
import os
import shutil
import pytesseract
from pdf2image import convert_from_path
import traceback

def find_tesseract_path():
    """Find Tesseract executable path"""
    possible_paths = [
        '/opt/homebrew/bin/tesseract',  # Apple Silicon
        '/usr/local/bin/tesseract',     # Intel Macs
        '/usr/bin/tesseract'            # System default
    ]
    
    # Check system PATH first
    path_from_which = shutil.which('tesseract')
    if path_from_which:
        return path_from_which
    
    # Check predefined paths
    for path in possible_paths:
        if os.path.exists(path):
            return path
    
    raise FileNotFoundError("Tesseract not found. Please install it with 'brew install tesseract'")

def extract_text_from_pdf(pdf_path, output_path=None, lang='hin'):
    try:
        # Dynamically find Tesseract path
        tesseract_path = find_tesseract_path()
        pytesseract.pytesseract.tesseract_cmd = tesseract_path
        
        # Rest of your existing script remains the same
        images = convert_from_path(pdf_path)
        
        full_text = ""
        for i, image in enumerate(images, 1):
            page_text = pytesseract.image_to_string(image, lang=lang)
            full_text += f"--- Page {i} ---\n{page_text}\n"
        
        if output_path:
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write(full_text)
            print(f"Text extracted and saved to {output_path}")
        
        return full_text
    
    except Exception as e:
        print(f"An error occurred: {e}")
        traceback.print_exc()
        return None

In [ ]:
# Example usage
if __name__ == "__main__":
    # pdf_path = "/Users/nitastha/Downloads/Fundamental Rules Hindi.pdf"
    # pdf_path = "/Users/nitastha/Desktop/Fundamental Rules Hindi - new.pdf"
    pdf_path = "/Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/Adobe Scan 12 Mar 2025 (1).pdf"
    output_path = "gst.txt"
    
    # Extract text
    extracted_text = extract_text_from_pdf(pdf_path, output_path)
    
    # If not saving to file, you can print the text
    if extracted_text:
        print(extracted_text)

In [1]:
with open("extracted_text.txt", "r", encoding="utf-8") as file:
    extracted_text = file.read()


len(extracted_text)

14732

In [57]:
# # Example usage
# if __name__ == "__main__":
#     # pdf_path = "/Users/nitastha/Downloads/Fundamental Rules Hindi.pdf"
#     pdf_path = "//Users/nitastha/Desktop/scannedpdf.pdf"
#     output_path = "extracted_text_scanned.txt"
    
#     # Extract text
#     extracted_text_scanned = extract_text_from_pdf(pdf_path, output_path)
    
#     # If not saving to file, you can print the text
#     if extracted_text_scanned:
#         print(extracted_text_scanned)

In [2]:
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Step 1: Read the input text file
file_path = "extracted_text.txt"
with open(file_path, "r", encoding="utf-8") as file:
    lines = file.readlines()

# Step 2: Process the text into LangChain Document format
documents = []
current_page = None
line_number = 0

for line in lines:
    line = line.strip()  # Remove leading/trailing whitespace
    if line.startswith("--- Page"):  # Detect page headers
        # Extract the page number from the header (e.g., '--- Page 1 ---')
        try:
            current_page = int(line.split()[-2])  # Get the page number (penultimate element)
        except ValueError:
            continue  # Skip if the page number is invalid (just in case)
        line_number = 0  # Reset line number for a new page
    elif line:  # Non-empty line
        documents.append(
            Document(
                page_content=line,
                metadata={"page": current_page, "line_number": line_number}
            )
        )
        line_number += 1

# Step 3: Split the documents using RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=70,
    length_function=len,
    is_separator_regex=False,
)

final_documents = text_splitter.split_documents(documents)

# Check the result
for doc in final_documents[:5]:  # Print the first 5 chunks for inspection
    print(doc)


page_content='करते हुए तथा मध्यप्रदेश सिवित्र सेवा (आचरण) नियम, 965 के नियम 24 के अधीन निर्वचन की' metadata={'page': 1, 'line_number': 0}
page_content='शक्तियों को उपयोग में लाते हुए, यह निर्देशित करता है कि-' metadata={'page': 1, 'line_number': 1}
page_content='(क) यदि आपात कारणों को छोड़कर, कोई भी शासकीय सेवक अनधिकृत रूप से' metadata={'page': 1, 'line_number': 2}
page_content='अनुपस्थित हो तो सक्षम प्राधिकारियों को आचरण नियमों के नियम 7 के अन्तर्गत' metadata={'page': 1, 'line_number': 3}
page_content='उपलब्ध अधिकारों का उपयोग करते समय सर्वप्रथम यह सुनिश्चित करना चाहिए कि' metadata={'page': 1, 'line_number': 4}


In [44]:
# from langchain.schema import Document
# from langchain.text_splitter import RecursiveCharacterTextSplitter

# # Step 1: Read the input text file
# file_path = "extracted_text.txt"
# with open(file_path, "r", encoding="utf-8") as file:
#     lines = file.readlines()

# # Step 2: Group the text by page
# documents = []
# current_page = None
# current_page_text = []

# for line in lines:
#     line = line.strip()  # Remove leading/trailing whitespace
#     if line.startswith("--- Page"):  # Detect page headers
#         # If we're starting a new page, save the current page's content
#         if current_page is not None and current_page_text:
#             documents.append(
#                 Document(
#                     page_content="\n".join(current_page_text),
#                     metadata={"page": current_page},
#                 )
#             )
#         # Extract the new page number
#         try:
#             current_page = int(line.split()[-2])  # Get the page number (penultimate element)
#         except ValueError:
#             continue  # Skip if the page number is invalid
#         current_page_text = []  # Reset the page text
#     elif line:  # Non-empty line
#         current_page_text.append(line)

# # Add the last page if there is content
# if current_page is not None and current_page_text:
#     documents.append(
#         Document(
#             page_content="\n".join(current_page_text),
#             metadata={"page": current_page},
#         )
#     )

# # Step 3: Split the documents using RecursiveCharacterTextSplitter
# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=1000,
#     chunk_overlap=50,
#     length_function=len,
#     is_separator_regex=False,
# )

# final_documents = text_splitter.split_documents(documents)

# # Check the result
# for doc in final_documents[:5]:  # Print the first 5 chunks for inspection
#     print(doc)


In [45]:
final_documents

[Document(metadata={'page': 1}, page_content='करते हुए तथा मध्यप्रदेश सिवित्र सेवा (आचरण) नियम, 965 के नियम 24 के अधीन निर्वचन की\nशक्तियों को उपयोग में लाते हुए, यह निर्देशित करता है कि-\n(क) यदि आपात कारणों को छोड़कर, कोई भी शासकीय सेवक अनधिकृत रूप से\nअनुपस्थित हो तो सक्षम प्राधिकारियों को आचरण नियमों के नियम 7 के अन्तर्गत\nउपलब्ध अधिकारों का उपयोग करते समय सर्वप्रथम यह सुनिश्चित करना चाहिए कि\nऐसी बिना सक्षम स्वीकृति के अनुपस्थित रहे शासकीय सेवक के विरुद्ध मध्यप्रदेश\nसिविल सेवा वर्गीकरण, नियंत्रण तथा अपील) नियम, 966 के अधीन अनुशासनिक\nकार्यवाही अविलम्ब प्रारंभ कर दी जाए |\n(ख) अनधिकृत अनुपस्थिति के पश्चात्\u200c जैसे ही ऐसा शासकीय सेवक कार्य पर उपस्थित\nहोने और इयूटी ज्वाइन करने के लिये उपस्थित हो तो उसे उपस्थित होने के दिनांक\nसे ही निलंबित किया जाय |\n(ग) कंडिका (क) में बताई गई अनुशासनात्मक कार्यवाही शीघ्रातिशीघ्र पूर्ण की जाए और\nउसमें यथोचित आदेश पारित किये जायें | उक्त आदेश पारित करने के साथ ही जैसी\nभी स्थिति हो, उसके मुताविक निलंबन अवधि के बारे मे भी निर्णय दिया जाए ।'),
 D

In [6]:
import fasttext as ft

# # # Download the FastText model
# !wget https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.hi.zip
# !unzip wiki.hi.zip

# Load the FastText model
embedding_model_path = '/Users/nitastha/Desktop/NitishFiles/Projects/wiki.hi/wiki.hi.bin'
embed_model = ft.load_model(embedding_model_path)

In [7]:
import pandas as pd

# convert the documents to a dataframe
# This dataframe will be used to create the embeddings
# And later will be used to update the Qdrant Vector Database
docs = final_documents
data = []
for doc in docs:
   # Get the page content and metadata for each chunk
   # Meta data contains chunk source or file name
   row_data = {
       "page_content": doc.page_content,
       "metadata": doc.metadata
   }
   data.append(row_data)

df = pd.DataFrame(data)

# Replace the new line characters with space
df['page_content'] = df['page_content'].replace('\\n', ' ', regex=True)

# Create a unique id for each document.
# This id will be used to update the Qdrant Vector Database
df['id'] = range(1, len(df) + 1)

# Create a payload column in the dataframe
# This payload column includes the page content and metadata
# This payload will be used when LLM needs to answer a query
df['payload'] = df[['page_content', 'metadata']].to_dict(orient='records')

# Create embeddings for each chunk
# This embeddings will be used when doing a similarity search with the user query
df['embeddings'] = df['page_content'].apply(lambda x: (embed_model.get_sentence_vector(x)).tolist())


In [8]:
df.head()

,page_content,metadata,id,payload,embeddings
0,करते हुए तथा मध्यप्रदेश सिवित्र सेवा (आचरण) नि...,{'page': 1},1,{'page_content': 'करते हुए तथा मध्यप्रदेश सिवि...,"[0.0031693181954324245, 0.007549724541604519, ..."
1,(घ) यहाँ यह ध्यान दिया जाए कि उपर्युक्त आदेश ल...,{'page': 1},2,{'page_content': '(घ) यहाँ यह ध्यान दिया जाए क...,"[-0.003495653858408332, 0.014883612282574177, ..."
2,रखा जाए तो उसके अनधिकृत अनुपस्थिति की अवधि का ...,{'page': 1},3,{'page_content': 'रखा जाए तो उसके अनधिकृत अनुप...,"[-0.0036470491904765368, 0.014627653174102306,..."
3,अनुदेश की ओर आकृष्ट करें तथा उसका इढतापूर्वक प...,{'page': 2},4,{'page_content': 'अनुदेश की ओर आकृष्ट करें तथा...,"[-0.001240079291164875, 0.016220491379499435, ..."
4,की जाती तो शासन टाली जा सकने वाली उलझनों और वि...,{'page': 2},5,{'page_content': 'की जाती तो शासन टाली जा सकने...,"[-0.008226384408771992, 0.017249753698706627, ..."


In [50]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, Batch


# Create a QdrantClient object
host = 'localhost'
port = 6333
client = QdrantClient(host=host, port=port)


In [53]:
# delete the collection if it already exists
client.delete_collection(collection_name="my_collection")

True

In [52]:


# Create a fresh collection in Qdrant
client.create_collection(
  collection_name="my_collection",
  vectors_config=VectorParams(size=300, distance=Distance.COSINE),
)

# Update the Qdrant Vector Database with the embeddings
# We are updating the embeddings in batches
# Since the data is large, we will only update the first batch of size 4000
batch_size = 4000
client.upsert(
collection_name="my_collection",
points=Batch(
    ids=df['id'].to_list()[:batch_size],
    payloads=df['payload'][:batch_size],
    vectors=df['embeddings'].to_list()[:batch_size],
),
)



UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [25]:
import mlflow
from qdrant_client import QdrantClient
mlflow.end_run()
mlflow_lgging = True

if mlflow_lgging:
   # set the experiment name in the mlflow
   mlflow.set_experiment("Hindi Chatbot")
   # start the mlflow run
   mlflow.start_run()

# load the Qdrant client from the same host and port
# this client will be used to interact with the Qdrant server
host = "localhost"
port = 6333
client = QdrantClient(host=host, port=port)

# log the parameters in the mlflow
if mlflow_lgging:
   mlflow.log_param("qdrant_host", host)
   mlflow.log_param("qdrant_port", port)

In [26]:
if mlflow_lgging:
   mlflow.log_param("embed_model_path", embedding_model_path)

In [27]:
from typing import List
from qdrant_client import QdrantClient
import fasttext as ft
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever

# Define a custom retriever class that uses Qdrant for document retrieval
# Since we're using FastText embeddings, we won't be able to use the default lanchain retriever, as it only supports HuggingFace and OpenAI Models
class QdrantRetriever(BaseRetriever):
   client: QdrantClient
   embed_model: ft.FastText._FastText
   collection_name: str
   limit: int

   def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun) -> List[Document]:
       """Converts query to a vector and retrieves relevant documents using Qdrant."""
       # Get the vector representation of the query using the FastText model
       query_vector = self.embed_model.get_sentence_vector(query).tolist()

       # Search for the most similar documents in the Qdrant collection
       # The search method returns a list of hits, where each hit contains the most similar document
       # we can limit the number of hits to return using the limit parameter
       search_results = self.client.search(
           collection_name=self.collection_name,
           query_vector=query_vector,
           limit=self.limit
       )
       # Finally, we convert the search results to a list of Document objects
       # that can be used by the pipeline
       return [Document(page_content=hit.payload['page_content']) for hit in search_results]

collection_name="my_collection"
limit = 500

# use the Custom QdrantRetriever class to create a retriever object
retriever = QdrantRetriever(
   client=client,
   embed_model=embed_model,
   collection_name=collection_name,
   limit=limit
)

if mlflow_lgging:
   mlflow.log_param("collection_name", collection_name)
   mlflow.log_param("limit", limit)

In [28]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

In [30]:
import config
from langchain_groq import ChatGroq
groq_api_key = config.GROQ_API_KEY
selected_model = "llama-3.1-70b-versatile"
llm = ChatGroq(groq_api_key=groq_api_key, model_name=selected_model)

# if mlflow_lgging:
#    mlflow.log_param("model_name", selected_model)

from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    """<s>[INST] आप एक विश्वसनीय और सटीक सहायक हैं। आपको केवल और केवल नीचे दिए गए संदर्भ के आधार पर प्रश्न का उत्तर देना है। 

निर्देश:
- केवल दिए गए संदर्भ से जानकारी का उपयोग करें
- यदि संदर्भ में उत्तर नहीं मिलता है, तो स्पष्ट रूप से कहें कि "दिए गए संदर्भ में इस प्रश्न का उत्तर नहीं मिलता"
- अपने ज्ञान या अतिरिक्त जानकारी को शामिल न करें
- उत्तर संक्षिप्त, स्पष्ट और सीधा होना चाहिए
- हिंदी भाषा में ही उत्तर दें

संदर्भ: {context} </s>
"""
)

prompt = ChatPromptTemplate.from_messages(
   [
       ("system", system_prompt),
       ("human", "{input}"),
   ]
)

# if mlflow_lgging:
#    mlflow.log_param("system_prompt", system_prompt)


question_answer_chain = create_stuff_documents_chain(llm, prompt)
chain = create_retrieval_chain(retriever, question_answer_chain)

In [23]:
query = 'राज्य सचिव आदेश 2 kya है?'

if mlflow_lgging:
   mlflow.log_param("query", query)

response = chain.invoke({"input": query})

# if mlflow_lgging:
#    mlflow.log_param("context", response['context'])
#    mlflow.log_param("response", response['answer'])

response

# end the logging of the mlflow
# mlflow.end_run()

NameError: name 'mlflow_lgging' is not defined

In [2]:
response

NameError: name 'response' is not defined

### gemini api

In [32]:
from langchain_google_genai import ChatGoogleGenerativeAI
import config


# Set up Gemini API
model_name = config.CHAT_MODEL  # Gemini Pro model
google_api_key = config.GOOGLE_API_KEY  # Replace with your actual Google API key

llm = ChatGoogleGenerativeAI(
    model=model_name,
    google_api_key=google_api_key,
    temperature=0.6,
    max_output_tokens=500
)

from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    """<s>[INST] आप एक विश्वसनीय और सटीक सहायक हैं। आपको केवल और केवल नीचे दिए गए संदर्भ के आधार पर प्रश्न का उत्तर देना है। 

निर्देश:
- केवल दिए गए संदर्भ से जानकारी का उपयोग करें
- यदि संदर्भ में उत्तर नहीं मिलता है, तो स्पष्ट रूप से कहें कि "दिए गए संदर्भ में इस प्रश्न का उत्तर नहीं मिलता"
- अपने ज्ञान या अतिरिक्त जानकारी को शामिल न करें
- उत्तर संक्षिप्त, स्पष्ट और सीधा होना चाहिए
- हिंदी भाषा में ही उत्तर दें

संदर्भ: {context} </s>
"""
)

prompt = ChatPromptTemplate.from_messages(
   [
       ("system", system_prompt),
       ("human", "{input}"),
   ]
)

# if mlflow_lgging:
#    mlflow.log_param("system_prompt", system_prompt)


question_answer_chain = create_stuff_documents_chain(llm, prompt)
chain = create_retrieval_chain(retriever, question_answer_chain)

In [1]:

query = 'मूल नियम ॥8 एवं मध्यप्रदेश सिवित्र सेवा (आचरण) नियम, 4965 के नियम 7 क्या कहता है विस्तार से बताओ?'

# if mlflow_lgging:
#    mlflow.log_param("query", query)

response = chain.invoke({"input": query})

# if mlflow_lgging:
#    mlflow.log_param("context", response['context'])
#    mlflow.log_param("response", response['answer'])

response

# end the logging of the mlflow
# mlflow.end_run()

NameError: name 'chain' is not defined

In [35]:
'अनधिकृत अनुपस्थिति कौन से नियम में बात हो रही है?'
response

{'input': 'मूल नियम ॥8 एवं मध्यप्रदेश सिवित्र सेवा (आचरण) नियम, 4965 के नियम 7 क्या कहता है विस्तार से बताओ?',
 'context': [Document(page_content='निकाल दिया गया\n\nअध्याय 4 - पदग्रहण काल\n\n05 से 08. निकाल दिया गया\n\nमध्यप्रदेश सिविल सेवायें (पदग्रहण काल) नियम, 982\n\nसंक्षिप्त नाम, प्रारंभ तथा त्रागू होना\nपरिभाषायें\n\n\n--- Page 8 ---\n(क) पदग्रहण काल\n\n(ख) स्थानान्तर\n\nजब किसी ऐसी सरकार या संगठन मे नियंत्रण के अधीन\nस्थानांतरण हो जिसने पदग्रहण काल की अवधि विहित\nकरने के लिए अपने नियम बनाये हैं\n\nपदग्रहण काल\n\nपदग्रहण काल का प्रारंभ तथा देय समय\n\nजब कोई शासकीय सेवक पूरे पदग्रहण काल का लाभ उठाये बिना नया पद ग्रहण करे\nपदग्रहण काल वेतन\n\nनिर्वचन\n\nनिरसन\n\nभाग 7\nअध्याय ॥2 - बाह्य सेवा\n\n09. 40.'),
  Document(page_content='भविष्य एवं अन्य निधियाँ\n7. तिथि जब से वेतन एवं भत्ते प्रभावशाली होंगे\n7-0. अनधिकृत गैर हाजिरी- क्\u200dया मानी जावे\n8._ लगातार अनुपस्थिति का प्रभाव\nशासकीय सेवकों की अनधिकृत अनुपस्थिति/ अनधिकृत अवकाश सा.प्र.वि.क्र.सी--6-38/92\nअनुशासनिक कार्यवाही राज्य 

In [33]:
query = 'राज्य सचिव आदेश 4 क्या कहता है विस्तार से बताओ?'
response = chain.invoke({"input": query})
response

{'input': 'राज्य सचिव आदेश 4 क्या कहता है विस्तार से बताओ?',
 'context': [Document(page_content='वेतन को छोड़कर) से अधिक नहीं होगा अथवा रुपये 0 प्रतिदिन, इन दोनों में से जो भी कम हो; (स) उपर्युक्त दो वर्षों की सीमा उन प्रकरणों में लागू नहीं होगी जहाँ विशेष वेतन भारतीय चिकित्सा सेवा के उन अधिकारियों को मंजूर किया गया है जो भारतीय रेलवे के कर्मचारियों की देखरेख कर रहे है । राज्य सचिव आदेश 4 - उन मामलों में जहाँ राज्य शासन ने भारतीय सिविल्र सेवा के अधिकारियों को मूल नियम 9 (2) (५) के अन्तर्गत व्यक्तिगत वेतन मंजूर किया है, ताकि भारतीय सिविल सेवा के समयमान में उन्हें जो हानि हुई है, उसकी पूर्ति हो सके जैसा कि वरिष्ठ सिवित्र सेवा नियमों की अनुसूची ४॥॥ में दिया गया है कि भारतीय सिविल सेवा के समयमान में मूल वेतन कम है । इस पर राज्य सचिव द्वारा यह मान्यता प्रदान की गई है कि मूल नियम 49 (2) (५) के उपबधों का अभिप्राय यह नहीं है कि इस नियम का इस प्रकार प्रयोग किया जाए और राज्य शासन की कार्यवाही जिससे मूल नियम 9 (2) (५) के अन्तर्गत व्यक्तिगत वेतन मंजूर किया जाना अनियमित है और इसका प्रभाव यह है कि ज

In [42]:
query = 'अनधिकृत अनुपस्थिति कौन से नियम में बात हो रही है?'
response = chain.invoke({"input": query})
response

{'input': 'अनधिकृत अनुपस्थिति कौन से नियम में बात हो रही है?',
 'context': [Document(page_content='की जाती तो शासन टाली जा सकने वाली उलझनों और विवादों से बचा रह सकता था और महीनों तक बिना काम की अवधि संबंधी वेतन इत्यादि के भुगतान करने की जिम्मेदारी से भी बच सकता था। 4. आशा की जाती है कि भविष्य में प्रश्नास्पद विहित अधिकारियों द्वारा पद 3 में उल्लेखित नियमों के पालन को अधिक इढता से सुनिश्चित करवायेंगे | इसके लिए यह जरूरी है कि आप अपनी उन मासिक बैठकों में जो आप नि:संदेह अपने कार्य क्षेत्र के अधीनस्थ अधिकारियों के साथ कार्य की मासिक विवेचना और क्क्ष्य निर्धारण के लिये बुलाते होंगे, स्पष्ट चर्चा करें और विषय के महत्व को अधीनस्थ अधिकारियों को नोट करायें | [सामान्य प्रशासन विभाग क्रमांक सी-3-/90/3/49, दिनांक 9-7-990| विषय- समयावधि । मध्यप्रदेश मूलभूत नियम भाग-एक के नियम 8 और भाग-दो के नियम 6 के तहत नियमानुसार शासन द्वारा निर्णय लिया जा चुका है- पूर्व में डाइस-नान के प्रकरण वित्त विभाग को भेजे जाते थे । राज्य शासन ने निर्णय लिया है कि अब डाइस-नान के प्रकरण प्रशासकीय विभागों द्वारा ही निपटाए जा

In [43]:
query = 'राज्य सचिव आदेश 2 क्या कहता है मुझे संक्षेप में बताओ'
response = chain.invoke({"input": query})
response

{'input': 'राज्य सचिव आदेश 2 क्या कहता है मुझे संक्षेप में बताओ',
 'context': [Document(page_content='दिन में रुपए 0 जो भी कम हो, से अधिक नहीं है, इस बात पर ध्यान दिये बिना कि विशेष वेतन अथवा व्यक्तिगत वेतन कितनी अवधि के लिए मंजूर किया जाना है, प्रांतीय सचिव की मंजूरी अपेक्षित होगी | महालेखा परीक्षक अनुदेश 4- नियम जो मूल नियम 22 एवं 23 को रदद नहीं करेंगे- मूल नियम 9 की यह मंशा नहीं है कि वह स्थानीय शासन को यह अधिकार देता है कि वह मूल नियम 22 एवं 23 में जितना वेतन अनुजैय है, उससे कम वेतन मंजूर करे | स्थानीय शासन नियम ।- पूर्व सैनिक सेवा की गणना जेल विभाग में वेतन वृद्धि के लिए की जाना- भूतपूर्व सैनिक जिसकी नियुक्ति मध्यप्रदेश के जेल विभाग में सहायक जेलर, मुख्य प्रहरी, एवं प्रहरी के रूप में की गई है, अथवा की जावेगी, वहाँ अनुमति है कि उसकी सेना की पूर्व सेवा'),
  Document(page_content='की जाती तो शासन टाली जा सकने वाली उलझनों और विवादों से बचा रह सकता था और महीनों तक बिना काम की अवधि संबंधी वेतन इत्यादि के भुगतान करने की जिम्मेदारी से भी बच सकता था। 4. आशा की जाती है कि भविष्य में प्रश्ना

In [40]:
import streamlit as st

# Ensure you have imported your chain correctly
# from your_module import chain

def generate_response(query):
    """
    Generate response using your existing RAG chain
    """
    try:
        # Make sure this matches how you typically invoke your chain
        response = chain.invoke({"input": query})
        return response
    except Exception as e:
        return f"An error occurred: {str(e)}"

def main():
    # Configure the page
    st.set_page_config(page_title="RAG Chat App", page_icon="💬")

    # App title
    st.title("Q&A Chat Interface")

    # Initialize session state for messages if not exists
    if 'messages' not in st.session_state:
        st.session_state.messages = []

    # Sidebar for any additional controls
    st.sidebar.header("Chat Settings")

    # Chat input
    user_input = st.chat_input("Enter your question here")

    # Process user input
    if user_input:
        # Add user message to chat history
        st.session_state.messages.append({
            "role": "user", 
            "content": user_input
        })

        # Display user message
        with st.chat_message("user"):
            st.write(user_input)

        # Generate and display response
        with st.chat_message("assistant"):
            with st.spinner("Generating response..."):
                response = generate_response(user_input)
                st.write(response)

        # Add assistant response to chat history
        st.session_state.messages.append({
            "role": "assistant", 
            "content": response
        })

    # Display chat history
    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.write(message["content"])

if __name__ == "__main__":
    main()

2024-12-13 13:50:06.491 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-12-13 13:50:06.492 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-12-13 13:50:06.493 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-12-13 13:50:06.494 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-12-13 13:50:06.494 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-12-13 13:50:06.495 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-12-13 13:50:06.496 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-12-13 13:50:06.497 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [41]:
!pip uninstall gradio

I0000 00:00:1734080214.642814  117551 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers


Found existing installation: gradio 4.44.0
Uninstalling gradio-4.44.0:
  Would remove:
    /Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/bin/gradio
    /Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/bin/upload_theme
    /Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/lib/python3.10/site-packages/gradio-4.44.0.dist-info/*
    /Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/lib/python3.10/site-packages/gradio/*
  Would not remove (might be manually added):
    /Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/lib/python3.10/site-packages/gradio/launches.json
Proceed (Y/n)? ^C
ERROR: Operation cancelled by user


In [39]:
import gradio as gr

def generate_response(message, history):
    """
    Generate response using your existing RAG chain
    
    Args:
    - message: Current user input
    - history: List of previous conversation turns
    """
    try:
        # Use only the current message for the query
        response = chain.invoke({"input": message})
        return response
    except Exception as e:
        return f"An error occurred: {str(e)}"

# Create Gradio interface
iface = gr.ChatInterface(
    fn=generate_response,
    title="RAG Chat Interface",
    description="Ask questions and get responses from your RAG model"
)

# Launch the interface
iface.launch(share=True)  # Added share=True in case you want to generate a public link

Running on local URL:  http://127.0.0.1:7861


/Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/lib/python3.10/site-packages/gradio/analytics.py:106: UserWarning: IMPORTANT: You are using gradio version 4.44.0, however version 4.44.1 is available, please upgrade. 
--------
  warnings.warn(
I0000 00:00:1734076293.161555  117551 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
Traceback (most recent call last):
  File "/Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/lib/python3.10/site-packages/gradio/queueing.py", line 536, in process_events
    response = await route_utils.call_process_api(
  File "/Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/lib/python3.10/site-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
  File "/Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/lib/python3.10/site-packages/gradio/blocks.py", line 1945, in process_api
    data = await self.postprocess_data(bloc


Could not create share link. Missing file: /Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/lib/python3.10/site-packages/gradio/frpc_darwin_amd64_v0.2. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.2/frpc_darwin_amd64
2. Rename the downloaded file to: frpc_darwin_amd64_v0.2
3. Move the file to this location: /Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/lib/python3.10/site-packages/gradio


Traceback (most recent call last):
  File "/Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/lib/python3.10/site-packages/gradio/queueing.py", line 536, in process_events
    response = await route_utils.call_process_api(
  File "/Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/lib/python3.10/site-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
  File "/Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/lib/python3.10/site-packages/gradio/blocks.py", line 1945, in process_api
    data = await self.postprocess_data(block_fn, result["prediction"], state)
  File "/Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/lib/python3.10/site-packages/gradio/blocks.py", line 1768, in postprocess_data
    prediction_value = block.postprocess(prediction_value)
  File "/Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/lib/python3.10/site-packages/gradio/components/chatbot.py", line 494, in p

In [37]:
!streamlit run /Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/lib/python3.10/site-packages/ipykernel_launcher.py

I0000 00:00:1734076079.051924  117551 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.29.242:8501

2024-12-13 13:18:05.389 MediaFileHandler: Missing file 6dfce643c1ce166b01e4302c5a5cbf95dd22bc71d8688da9d8a51f44.jpg
NOTE: When using the `ipython kernel` entry point, Ctrl-C will not work.

To exit, you will have to explicitly quit this process, by either sending
"quit" from a client, or using Ctrl-\ in UNIX-like environments.

To read more about this, see https://github.com/ipython/ipython/issues/2049


To connect another client to this kernel, use:
    --existing kernel-31427.json
[IPKernelApp] ERROR | Unable to initialize signal:
Traceback (most recent call last):
  File "/Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/lib/python3.10/site-packages/ipykernel/kernelapp.py", line 701, in initialize
    self.init_signal()
  File "/Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/lib/python3.10/site-packages/ipykernel/kernelapp.py", 

In [8]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

    
    # if is_directory:
    #     loader = PyPDFDirectoryLoader(pdf_path)
    # else:
    #     loader = PyPDFLoader(pdf_path)
    
    # documents = loader.load()
    # text_splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=1000)
    # final_documents = text_splitter.split_documents(documents)
# vector_store = FAISS.from_documents(final_documents, embeddings)

    

In [9]:
from langchain_experimental.text_splitter import SemanticChunker
text_splitter = SemanticChunker(GoogleGenerativeAIEmbeddings(model="models/embedding-001"))

with open("extracted_text_full.txt") as f:
    raw_text = f.read()
# docs = text_splitter.create_documents([raw_text])
# print(docs[0].page_content)

In [10]:
text_splitter = SemanticChunker(
    GoogleGenerativeAIEmbeddings(model="models/embedding-001"), breakpoint_threshold_type="percentile"
)
docs_percentile = text_splitter.create_documents([raw_text])
print(len(docs_percentile),"---percentile------")
text_splitter = SemanticChunker(
    GoogleGenerativeAIEmbeddings(model="models/embedding-001"), breakpoint_threshold_type="standard_deviation"
)
docs_standard_deviation = text_splitter.create_documents([raw_text])
print(len(docs_standard_deviation),"---standard_deviation------")
text_splitter = SemanticChunker(
    GoogleGenerativeAIEmbeddings(model="models/embedding-001"), breakpoint_threshold_type="interquartile"
)
docs_interquartile = text_splitter.create_documents([raw_text])
print(len(docs_interquartile),"---interquartile------")
text_splitter = SemanticChunker(
    GoogleGenerativeAIEmbeddings(model="models/embedding-001"), breakpoint_threshold_type="gradient"
)
docs_gradient = text_splitter.create_documents([raw_text])
print(len(docs_gradient),"---gradient------")
# print(docs[0].page_content)


65 ---percentile------
33 ---standard_deviation------
94 ---interquartile------
65 ---gradient------


In [8]:

print(len(docs_percentile),"---percentile------")
print(len(docs_standard_deviation),"---standard_deviation------")
print(len(docs_interquartile),"---interquartile------")
print(len(docs_gradient),"---gradient------")



65 ---percentile------
33 ---standard_deviation------
94 ---interquartile------
65 ---gradient------


In [41]:
from langchain_google_genai import ChatGoogleGenerativeAI
import config


# Set up Gemini API
model_name = config.CHAT_MODEL  # Gemini Pro model
google_api_key = config.GOOGLE_API_KEY  # Replace with your actual Google API key

llm = ChatGoogleGenerativeAI(
    model=model_name,
    google_api_key=google_api_key,
    temperature=0.7,
    max_output_tokens=100
)

from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    """<s>[INST] आप एक विश्वसनीय और सटीक सहायक हैं। आपको केवल और केवल नीचे दिए गए संदर्भ के आधार पर प्रश्न का उत्तर देना है। 

निर्देश:
- केवल दिए गए संदर्भ से जानकारी का उपयोग करें
- यदि संदर्भ में उत्तर नहीं मिलता है, तो स्पष्ट रूप से कहें कि "दिए गए संदर्भ में इस प्रश्न का उत्तर नहीं मिलता"
- अपने ज्ञान या अतिरिक्त जानकारी को शामिल न करें
- उत्तर संक्षिप्त, स्पष्ट और सीधा होना चाहिए
- हिंदी भाषा में ही उत्तर दें

संदर्भ: {context} </s>
"""
)

prompt = ChatPromptTemplate.from_messages(
   [
       ("system", system_prompt),
       ("human", "{input}"),
   ]
)

question_answer_chain = create_stuff_documents_chain(llm, prompt)
chain = create_retrieval_chain(retriever, question_answer_chain)

vector_store = FAISS.from_documents(docs_standard_deviation, embeddings)
# prompt = ChatPromptTemplate.from_template(
#     "You are a helpful AI assistant. Answer questions based solely on the provided context. If the answer is not in the context, say 'The answer is not in the provided context.' Do not add any additional commentary or information beyond what is given.:\n\n{context}\n\nQuestion: {input}\nAnswer:"
# )


document_chain = create_stuff_documents_chain(llm, prompt)
retriever = vector_store.as_retriever()
retrieval_chain = create_retrieval_chain(retriever, document_chain)
prompt1 = 'मूल नियम ॥8 एवं मध्यप्रदेश सिवित्र सेवा (आचरण) नियम, 4965 के नियम 7 क्या कहता है?'

# Get the response and store it in the dictionary with the document name as the key
print(retrieval_chain.invoke({"input": prompt1}))


{'input': 'मूल नियम ॥8 एवं मध्यप्रदेश सिवित्र सेवा (आचरण) नियम, 4965 के नियम 7 क्या कहता है?', 'context': [Document(page_content='निकाल दिया गया\n\nअध्याय 4 - पदग्रहण काल\n\n05 से 08. निकाल दिया गया\n\nमध्यप्रदेश सिविल सेवायें (पदग्रहण काल) नियम, 982\n\nसंक्षिप्त नाम, प्रारंभ तथा त्रागू होना\nपरिभाषायें\n\n\n--- Page 8 ---\n(क) पदग्रहण काल\n\n(ख) स्थानान्तर\n\nजब किसी ऐसी सरकार या संगठन मे नियंत्रण के अधीन\nस्थानांतरण हो जिसने पदग्रहण काल की अवधि विहित\nकरने के लिए अपने नियम बनाये हैं\n\nपदग्रहण काल\n\nपदग्रहण काल का प्रारंभ तथा देय समय\n\nजब कोई शासकीय सेवक पूरे पदग्रहण काल का लाभ उठाये बिना नया पद ग्रहण करे\nपदग्रहण काल वेतन\n\nनिर्वचन\n\nनिरसन\n\nभाग 7\nअध्याय ॥2 - बाह्य सेवा\n\n09. 40.'), Document(page_content='ले. अधि. 9300-34800 3200/-\n04.02.2007 44580/- 3200/- केवल ग्रेड पे दी गई |\n04.07.2007 2390/- 3200/- एक वार्षिक वेतनवृद्धि पदोन्\u200dनति\n\nके पूर्व मूल वेतन 580-2800\nका 3 प्रतिशत जोड़कर 42020/-\nका 3 प्रतिशत 370 जोड़कर रु. 2390/- निर्धारित होगा |\n\nउदाहरण-4. एक सहायक ग्

In [38]:
responses = {}  # Dictionary to store responses for each document type
import config
from langchain_groq import ChatGroq
groq_api_key = config.GROQ_API_KEY
selected_model = "llama-3.1-8b-instant"
llm = ChatGroq(groq_api_key=groq_api_key, model_name=selected_model)
for final_documents in [docs_standard_deviation]:
    # Get the name of the current document type to use as the key
    document_name = final_documents.__name__ if hasattr(final_documents, '__name__') else str(final_documents)
        vector_store = FAISS.from_documents(final_documents, embeddings)
    prompt = ChatPromptTemplate.from_template(
        "You are a helpful AI assistant. Answer questions based solely on the provided context. If the answer is not in the context, say 'The answer is not in the provided context.' Do not add any additional commentary or information beyond what is given.:\n\n{context}\n\nQuestion: {input}\nAnswer:"
    )


    document_chain = create_stuff_documents_chain(llm, prompt)
    retriever = vector_store.as_retriever()
    retrieval_chain = create_retrieval_chain(retriever, document_chain)

    prompt1 = 'मूल नियम ॥8 एवं मध्यप्रदेश सिवित्र सेवा (आचरण) नियम, 4965 के नियम 7 क्या कहता है?'

    # Get the response and store it in the dictionary with the document name as the key
    print(retrieval_chain.invoke({"input": prompt1}))
    responses[f'response_{document_name}'] = retrieval_chain.invoke({"input": prompt1})
    # Create the vector store and the retriever chain


# Example to check the response for a specific document type
print(responses.get('response_docs_percentile'))  # Adjust based on how you handle document names


APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01hs1sp1bfftj9dhthfs18jhhf` on tokens per minute (TPM): Limit 20000, Requested 29542, please reduce your message size and try again. Visit https://console.groq.com/docs/rate-limits for more information.', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [112]:
prompt = ChatPromptTemplate.from_template(
    "You are a helpful AI assistant. Answer questions based solely on the provided context. If the answer is not in the context, say 'The answer is not in the provided context.' Do not add any additional commentary or information beyond what is given.:\n\n{context}\n\nQuestion: {input}\nAnswer:"
)

llm = ChatGroq(groq_api_key=groq_api_key, model_name='llama3-groq-70b-8192-tool-use-preview')
document_chain = create_stuff_documents_chain(llm, prompt)
retriever = vector_store.as_retriever()
retrieval_chain = create_retrieval_chain(retriever, document_chain)


prompt1 = 'मूल नियम ॥8 एवं मध्यप्रदेश सिवित्र सेवा (आचरण) नियम, 4965 के नियम 7 क्या कहता है?'
# start = time.time()
response = retrieval_chain.invoke({"input": prompt1})


In [113]:
response

{'input': 'मूल नियम ॥8 एवं मध्यप्रदेश सिवित्र सेवा (आचरण) नियम, 4965 के नियम 7 क्या कहता है?',
 'context': [Document(metadata={'page': 2}, page_content='अनुदेश की ओर आकृष्ट करें तथा उसका इढतापूर्वक पालन कराये |\n[सामान्य प्रशासन विभाग क्रमांक 62/464/(3)/79, दिनांक 28--980]\nशासन का ध्यान कुछ ऐसे प्रकरणों की ओर आकृष्ट किया गया है जिसमें विभागों के\nकतिपय शासकीय सेवकों ने अपने कर्त्तव्य से अनधिकृत रूप से अनुपस्थित रहने के पश्चात्\u200c संबंधित\nविहित प्राधिकारी को अपना कार्यभार ग्रहण करने का प्रतिवेदन प्रस्तुत किया, किन्तु संबंधित विहित\nप्राधिकारी द्वारा उपर्युक्त स्वरूप वाले कार्यभार ग्रहण करने के प्रतिवेदन स्वीकार नहीं किए गए ।\n2. फलस्वरूप अनधिकृत रूप से अनुपस्थित (अवकाश) पर रहने के पश्चात्\u200c संबंधित विहित\nप्राधिकारियों द्वारा उनके कार्यभार ग्रहण करने की सूचना को स्वीकार नहीं किये जाने की कार्यवाही\nके लिये न्\u200dयायात्रय/प्रशासनिक अधिकरण में याचिकाएँ प्रस्तुत की गई ।\nमूल नियम ॥8 एवं मध्यप्रदेश सिवित्र सेवा (आचरण) नियम, 4965 के नियम 7 में अवकाश\nसे संबंधित प्रावधान हैं | यदि 

In [116]:
pip install flair

I0000 00:00:1733212962.432637 8102423 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers



Usage:   
  /Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/bin/python -m pip <command> [options]

no such option: --user
Note: you may need to restart the kernel to use updated packages.


In [2]:
from typing import List
def semantic_splitter(text: str, chunk_size: int = 100, chunk_overlap: int = 50) -> List[str]:
    from flair.models import SequenceTagger
    from flair.data import Sentence
    from flair.splitter import SegtokSentenceSplitter

    splitter = SegtokSentenceSplitter()
    
    # Split text into sentences
    sentences = splitter.split(text)

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        # Add sentence to the current chunk
        if len(current_chunk) + len(sentence.to_plain_string()) <= chunk_size:
            current_chunk += " " + sentence.to_plain_string()
        else:
            # If adding the next sentence exceeds max size, start a new chunk
            chunks.append(current_chunk.strip())
            current_chunk = sentence.to_plain_string()

    # Add the last chunk if it exists
    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

In [6]:
semantic_splitter(extracted_text, chunk_size=512,chunk_overlap=212)

/Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2024-12-09 20:58:38,675 Warning: An empty Sentence was created! Are there empty strings in your dataset?


['--- Page 1 --- करते हुए तथा मध्यप्रदेश सिवित्र सेवा (आचरण) नियम, 965 के नियम 24 के अधीन निर्वचन की शक्तियों को उपयोग में लाते हुए, यह निर्देशित करता है कि-',
 '(क) यदि आपात कारणों को छोड़कर, कोई भी शासकीय सेवक अनधिकृत रूप से अनुपस्थित हो तो सक्षम प्राधिकारियों को आचरण नियमों के नियम 7 के अन्तर्गत उपलब्ध अधिकारों का उपयोग करते समय सर्वप्रथम यह सुनिश्चित करना चाहिए कि ऐसी बिना सक्षम स्वीकृति के अनुपस्थित रहे शासकीय सेवक के विरुद्ध मध्यप्रदेश सिविल सेवा वर्गीकरण, नियंत्रण तथा अपील) नियम, 966 के अधीन अनुशासनिक कार्यवाही अविलम्ब प्रारंभ कर दी जाए |',
 '(ख) अनधिकृत अनुपस्थिति के पश्चात् जैसे ही ऐसा शासकीय सेवक कार्य पर उपस्थित होने और इयूटी ज्वाइन करने के लिये उपस्थित हो तो उसे उपस्थित होने के दिनांक से ही निलंबित किया जाय | (ग) कंडिका (क) में बताई गई अनुशासनात्मक कार्यवाही शीघ्रातिशीघ्र पूर्ण की जाए और उसमें यथोचित आदेश पारित किये जायें | उक्त आदेश पारित करने के साथ ही जैसी भी स्थिति हो, उसके मुताविक निलंबन अवधि के बारे मे भी निर्णय दिया जाए ।',
 '(घ) यहाँ यह ध्यान दिया जाए कि उपर्युक्त आ